# Phase 3c: fine-tune Qwen3.5-2B (non-thinking mode) on v1.2

**Why Qwen now.** On LFM2.5-1.2B-Thinking, training on short teacher answers (`v1.2`) broke its long thinking (UGPhysics 13.3% → 5.5%), and training on its own answers (`self2`) gave no gain (10.9%). Qwen3.5-2B in non-thinking mode already scores 17.5% on the same 260 questions, and it writes short visible reasoning, the same style as `v1.2`. Its main weakness is format: about half of its answers run past 4,096 tokens and 42% have no boxed answer. Training on `v1.2` should fix the format and give it physics practice.

**Who does what**

| Account | Task | Sections |
|---|---|---|
| A | data check + speed test (about 40 min), then send me the numbers | 1 to 5 |
| C (at the same time) | Qwen baseline on MATH-500, once (about 30 min) | 1 to 3, then 9a |
| A, then B | full training, after I confirm `MAX_TRAIN` and `GRAD_CKPT` | 1 to 3, then 6 |
| C | evaluate kept checkpoints | 1 to 3, then 9b |

**Before you start**
1. Put the new `phystrain.py` (**version 1.4**) in the Drive folder and replace the old one. `physeval.py` stays at 1.3.
2. **Runtime > Change runtime type > T4 GPU.**
3. Everything saves to Drive. After a disconnect, run sections 1 to 3, then the cell you were on.

## 1. Settings

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
WORKDIR = "/content/drive/MyDrive/physics_eval"
MODEL = "Qwen/Qwen3.5-2B"       # LFM runs used "LiquidAI/LFM2.5-1.2B-Thinking"
NO_THINKING = True              # Qwen non-thinking mode: plain-text reasoning, then the boxed answer
DATA = "v1.2"                   # short step-by-step answers (PhysicsData.ipynb)
PEFT = "lora"
RUN_NAME = f"qwen-{DATA}-{PEFT}"  # results go to WORKDIR/train_runs/<RUN_NAME>

RANK, ALPHA, DROPOUT = 32, 64, 0.0
LR = 1e-4
EPOCHS = 1
MAX_TRAIN = 12000               # training examples used (0 = all 29,500); about 4 to 5 GPU hours
EXAMPLES_PER_STEP = 16
MAX_LEN = 2048                  # prompt + answer, in Qwen tokens
TOKENS_PER_MICRO = 8192         # tokens per forward pass. Speed test: 5.0 GB peak at 2048, so bigger passes fit
GRAD_CKPT = True                # speed test: without it, out of GPU memory
HEALTH_MAX_NEW = 2048           # token budget of the generation check
KEEP_EVERY = 250                # keep an evaluable checkpoint (and run the generation check) every N steps
VAL_EVERY = 100                 # validation loss every N steps

MODEL_FLAGS = f"--model {MODEL}" + (" --no-thinking" if NO_THINKING else "")
CKPT_FLAG = "" if GRAD_CKPT else "--no-grad-ckpt"
TRAIN_ARGS = (f'--workdir "{WORKDIR}" --data {DATA} {MODEL_FLAGS} --peft {PEFT} --rank {RANK} --alpha {ALPHA} '
              f'--dropout {DROPOUT} --lr {LR} --epochs {EPOCHS} --max-train {MAX_TRAIN} '
              f'--examples-per-step {EXAMPLES_PER_STEP} --tokens-per-micro {TOKENS_PER_MICRO} --max-len {MAX_LEN} '
              f'--health-max-new {HEALTH_MAX_NEW}')
EVAL_FLAGS = f"--model {MODEL}" + (" --enable-thinking false" if NO_THINKING else "")
BASE_RUN = "qwen35-2b-nothink"  # the base model's evaluation run, for comparison
print(TRAIN_ARGS)

## 2. Mount Drive, check the GPU and the files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys
need = ["phystrain.py", "physeval.py", f"train_data/{DATA}/train.jsonl", f"train_data/{DATA}/val.jsonl"]
for f in need:
    assert os.path.exists(f"{WORKDIR}/{f}"), f"missing {f}"
sys.path.insert(0, WORKDIR)
import physeval, phystrain
assert physeval.VERSION >= "1.3", "replace physeval.py with version 1.3"
assert phystrain.VERSION >= "1.4", "replace phystrain.py with version 1.4"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Install packages

In [ ]:
!pip -q uninstall -y torchao   # Colab's torchao 0.10 makes new PEFT fail ("incompatible version of torchao"); not needed
!pip -q install -U transformers peft accelerate safetensors
!python -c "import torch, transformers, peft; print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)"

## 4. Check the data and the format (3 to 5 minutes)
Read the output. It must show:
- `examples without a thinking trace: 0` and `boxed-style examples without a boxed answer: 0`
- `think format (no thinking section: reasoning is trained as plain text)`
- **NOT trained (prompt)**: ends with `<|im_start|>assistant`, then an empty `<think>` `</think>` block. That is how Qwen's non-thinking mode looks, and the evaluation uses the same prompt. No "Liquid AI" anywhere.
- **TRAINED**: starts straight with the reasoning (no `<think>`), ends with `So the final answer is \boxed{...}` and `<|im_end|>`
- `| N too long` at the end of the token line: small (under about 60 of 3,000)
- adapter layers: `q_proj, k_proj, v_proj, o_proj` (6 each), `in_proj_qkv, in_proj_z, out_proj` (18 each), `gate_proj, up_proj, down_proj` (24 each)

In [ ]:
!python "{WORKDIR}/phystrain.py" check --workdir "{WORKDIR}" --data {DATA} {MODEL_FLAGS} --max-len {MAX_LEN} --load-model

## 5. Speed test (about 30 minutes, account A)
Two short runs of 20 training steps on the real data. Their run names end in `-speedA` / `-speedB`, so they never mix with the real run.
- **A**: with gradient checkpointing (less memory, more compute)
- **B**: without it (faster, if it fits in memory)

The first run downloads the model (a few GB). These messages are expected and harmless:
- `chunk_gated_delta_rule is falling back to its reference PyTorch implementation because flash-linear-attention is not installed` (a slower but correct version of Qwen's linear-attention layers; the speed test tells us if it matters)
- the `causal_conv1d` warning

**Send me** from each run: the `usable (... dropped)` line, the `plan:` line, and the last `step 20/...` line (`loss`, `s/step`, `peak mem`, `skipped`). From run A also the `validation loss` line. If run B stops with `Out of GPU memory`, that is fine; just tell me.

Rerunning a speed test with the same name does nothing (it is already at step 20). To repeat one, add a number to the name, e.g. `-speedA2`.

In [ ]:
!python "{WORKDIR}/phystrain.py" train {TRAIN_ARGS} --run-name {RUN_NAME}-speedA \
    --max-steps 20 --log-every 5 --val-every 20 --save-every 100000 --health-every 0 --keep-every 0

In [ ]:
!python "{WORKDIR}/phystrain.py" train {TRAIN_ARGS} --run-name {RUN_NAME}-speedB \
    --max-steps 20 --log-every 5 --val-every 100000 --save-every 100000 --health-every 0 --keep-every 0 --no-grad-ckpt

### 5b. How long will the full run take?
Put the faster `s/step` that fitted in memory (run B if it fitted with peak mem below about 13.5 GB, otherwise run A). The validation and generation checks add roughly 10% on top. I'll confirm `MAX_TRAIN` from your numbers; then set it (and `GRAD_CKPT`) in section 1 and **run section 1 again**.

In [ ]:
import math
S_PER_STEP = 21.0   # <- from the speed test
for n in (6000, 9000, 12000, 18000, 29500):
    steps = math.ceil(n / EXAMPLES_PER_STEP) * EPOCHS
    print(f"MAX_TRAIN {n:>6}: {steps:>5} steps -> about {steps * S_PER_STEP / 3600:.1f} GPU hours")

## 6. Full training (the long step)
Only after the speed test, with `MAX_TRAIN` and `GRAD_CKPT` set in section 1 (and section 1 run again).

**Check the first two log lines** (steps 10 and 20): `peak mem` should be below about 12 GB. `skipped` is about 5 at the start (the fp16 loss scale is being found, normal) and then should stop rising. If it stops with `Out of GPU memory`, set `TOKENS_PER_MICRO = 4096` and `RUN_NAME = f"qwen-{DATA}-{PEFT}-m4k"`, run section 1 again, then this cell.

It prints loss, learning rate, seconds per step and an ETA every 10 steps, and validation loss every 100 steps. Every 250 steps it keeps an evaluable checkpoint in `train_runs/<RUN_NAME>/ckpt-XXXXX` and runs the generation check on 8 validation questions (also once before training, to record the base model). In that check `closed_thinking` is always 1.0 for Qwen (there is no thinking section). Watch `boxed_after_thinking` (here: the answer has a boxed result) and `stopped`; both should rise towards 1.0, and `mean_tokens` should drop far below the base model's.

**Safe to interrupt.** Rerun this cell (after sections 1 to 3) to continue from the last save (every 200 steps and at every kept checkpoint).

**Two accounts, one run:** when one account's GPU time runs out, another account continues **the same run**: same section 1 settings, run sections 1 to 3, then this cell. It prints `RESUMED from step ...`. Never run the same `RUN_NAME` on two accounts at the same time. `GRAD_CKPT` may differ between sessions; every other setting must stay the same (the script refuses otherwise).

In [ ]:
!python "{WORKDIR}/phystrain.py" train {TRAIN_ARGS} --run-name {RUN_NAME} \
    --keep-every {KEEP_EVERY} --health-every {KEEP_EVERY} --val-every {VAL_EVERY} {CKPT_FLAG}

## 7. Training curves
- **train loss** should fall quickly at first, then slowly.
- **validation loss** should fall and then flatten. If it rises for two checks in a row while train loss keeps falling, the model is over-fitting; the checkpoint just before the rise is the one to evaluate.
- **health**: `boxed_after_thinking` and `stopped` should go up towards 1.0; `mean tokens` should drop far below the base model's (step 0).

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f"{WORKDIR}/train_runs/{RUN_NAME}/log.csv")
tr, va, he = log[log.train_loss.notna()], log[log.val_loss.notna()], log[log.closed_thinking.notna()]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(tr.step, tr.train_loss, label="train"); ax[0].plot(va.step, va.val_loss, "o-", label="validation")
ax[0].set_title("loss"); ax[0].set_xlabel("step"); ax[0].legend()
ax[1].plot(tr.step, tr.lr); ax[1].set_title("learning rate"); ax[1].set_xlabel("step")
for c in ["boxed_after_thinking", "stopped"]:
    ax[2].plot(he.step, he[c], "o-", label=c)
ax[2].set_ylim(0, 1.05); ax[2].set_title("generation health"); ax[2].set_xlabel("step"); ax[2].legend()
plt.show()
print(he[["step", "boxed_after_thinking", "stopped", "mean_tokens"]].to_string(index=False))

## 8. Read the latest generated answers

In [ ]:
import json
rows = [json.loads(l) for l in open(f"{WORKDIR}/train_runs/{RUN_NAME}/samples.jsonl")]
last = max(r["step"] for r in rows)
for r in [r for r in rows if r["step"] == last][:3]:
    print("=" * 100, f"\nstep {r['step']} | finished: {r['finished']} | reference: {r['reference']}\n")
    print(r["output"][-2500:])

## 9a. Qwen baseline on MATH-500 (done)
We have Qwen's UGPhysics baseline but not its MATH-500 score. This adds the same 70 MATH-500 questions the LFM runs answered (10 per subject), so we can see whether training hurts maths. Same settings as the existing `qwen35-2b-nothink` run; its UGPhysics answers are kept.

In [ ]:
EVAL_NAME = BASE_RUN
!pip -q uninstall -y torchao
!pip -q install "antlr4-python3-runtime==4.11.1" chardet
!python "{WORKDIR}/physeval.py" generate --workdir "{WORKDIR}" --run {BASE_RUN} \
    --benchmarks math500 --per-group "math500=10" --max-new-tokens 4096 --batch-size 16
!python "{WORKDIR}/physeval.py" grade --workdir "{WORKDIR}" --run {BASE_RUN}
!python "{WORKDIR}/physeval.py" report --workdir "{WORKDIR}" --run lfm25-1.2b-thinking,{BASE_RUN} > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{WORKDIR}/results/summary.md").read()))
import shutil; shutil.copy(f"{WORKDIR}/results/summary.md", f"{WORKDIR}/results/summary_{EVAL_NAME}.md")
print("saved a copy: results/summary_" + EVAL_NAME + ".md")

## 9b. Evaluate a checkpoint (account C, or any free account)
Same harness and settings as the baselines (greedy, 4,096 tokens, Qwen non-thinking prompt), on the questions the base model already answered: 20 UGPhysics questions per subject and 10 MATH-500 per subject. The report compares it with the base Qwen and LFM runs.

Set `STEP` to a kept checkpoint (250, 500, ...). Send me `results/summary.md`.

In [ ]:
EVAL_RUN, STEP = RUN_NAME, 250
ADAPTER = f"{WORKDIR}/train_runs/{EVAL_RUN}/ckpt-{STEP:05d}"
EVAL_NAME = f"ft-{EVAL_RUN}-s{STEP}"
assert os.path.exists(f"{ADAPTER}/adapter_config.json"), "checkpoint not found yet"
!pip -q uninstall -y torchao
!pip -q install "antlr4-python3-runtime==4.11.1" chardet
!python "{WORKDIR}/physeval.py" generate --workdir "{WORKDIR}" --run {EVAL_NAME} {EVAL_FLAGS} --adapter "{ADAPTER}" \
    --benchmarks ugphysics,math500 --per-group "ugphysics=20,math500=10" --max-new-tokens 4096 --batch-size 16
!python "{WORKDIR}/physeval.py" grade --workdir "{WORKDIR}" --run {EVAL_NAME}
!python "{WORKDIR}/physeval.py" report --workdir "{WORKDIR}" --run lfm25-1.2b-thinking,{BASE_RUN},{EVAL_NAME} > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{WORKDIR}/results/summary.md").read()))
import shutil; shutil.copy(f"{WORKDIR}/results/summary.md", f"{WORKDIR}/results/summary_{EVAL_NAME}.md")
print("saved a copy: results/summary_" + EVAL_NAME + ".md")

## 9c. Bigger physics check (about 3 to 4 GPU hours)
260 questions are too few to see a gain of 2 points. Base Qwen has already answered 1,008 dev questions, so this answers the same ones with an adapter; base Qwen is not run again. The 260 already done are reused. Safe to interrupt; rerun the cell (any account, but only one at a time) to continue.

`STRENGTH = 1.0` is the trained adapter (done: +2.0 points). `STRENGTH = 0.5` is the half-strength copy from 9d (run 9d for that strength first). Send me the saved `results/summary_<name>.md`.

In [ ]:
STRENGTH = 0.5                  # 1.0 = trained adapter (done), 0.5 = half strength
EVAL_RUN, STEP = RUN_NAME, 647
SRC = f"{WORKDIR}/train_runs/{EVAL_RUN}/ckpt-{STEP:05d}"
TAG = "" if STRENGTH == 1.0 else f"-x{int(STRENGTH * 100)}"
ADAPTER, EVAL_NAME = SRC + TAG, f"ft-{EVAL_RUN}-s{STEP}" + TAG
assert os.path.exists(f"{ADAPTER}/adapter_config.json"), "adapter not found (for STRENGTH < 1, run 9d first)"
print("evaluating", EVAL_NAME)
!pip -q uninstall -y torchao
!pip -q install "antlr4-python3-runtime==4.11.1" chardet
!python "{WORKDIR}/physeval.py" generate --workdir "{WORKDIR}" --run {EVAL_NAME} {EVAL_FLAGS} --adapter "{ADAPTER}" \
    --benchmarks ugphysics --per-group "ugphysics=100" --max-new-tokens 4096 --batch-size 16
!python "{WORKDIR}/physeval.py" grade --workdir "{WORKDIR}" --run {EVAL_NAME}
!python "{WORKDIR}/physeval.py" report --workdir "{WORKDIR}" --run {BASE_RUN},ft-{EVAL_RUN}-s{STEP},ft-{EVAL_RUN}-s{STEP}-x50 > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{WORKDIR}/results/summary.md").read()))
import shutil; shutil.copy(f"{WORKDIR}/results/summary.md", f"{WORKDIR}/results/summary_{EVAL_NAME}.md")
print("saved a copy: results/summary_" + EVAL_NAME + ".md")

## 9d. Half-strength adapter (done for 0.5: maths back to 69%)
The trained answers are short (median about 500 tokens), which fixed the format but hurt hard maths (MATH Level 5: 40% to 13%). This copies `ckpt-00647` and applies only half of the change, a point between base Qwen and the trained model. No training needed: it halves `lora_alpha` (64 to 32) in a copy of the adapter. Send me the `summary.md` it shows.

In [ ]:
import json, shutil
STRENGTH = 0.5
SRC = f"{WORKDIR}/train_runs/{RUN_NAME}/ckpt-00647"
ADAPTER = f"{SRC}-x{int(STRENGTH * 100)}"
if not os.path.exists(f"{ADAPTER}/adapter_config.json"):
    shutil.copytree(SRC, ADAPTER)
    cfg = json.load(open(f"{ADAPTER}/adapter_config.json"))
    assert not cfg.get("use_rslora") and not cfg.get("alpha_pattern"), "unexpected adapter settings; tell me"
    cfg["lora_alpha"] = int(round(cfg["lora_alpha"] * STRENGTH))
    json.dump(cfg, open(f"{ADAPTER}/adapter_config.json", "w"), indent=2)
print("lora_alpha now", json.load(open(f"{ADAPTER}/adapter_config.json"))["lora_alpha"], "(the trained adapter has 64)")
EVAL_NAME = f"ft-{RUN_NAME}-s647-x{int(STRENGTH * 100)}"
!pip -q uninstall -y torchao
!pip -q install "antlr4-python3-runtime==4.11.1" chardet
!python "{WORKDIR}/physeval.py" generate --workdir "{WORKDIR}" --run {EVAL_NAME} {EVAL_FLAGS} --adapter "{ADAPTER}" \
    --benchmarks ugphysics,math500 --per-group "ugphysics=20,math500=10" --max-new-tokens 4096 --batch-size 16
!python "{WORKDIR}/physeval.py" grade --workdir "{WORKDIR}" --run {EVAL_NAME}
!python "{WORKDIR}/physeval.py" report --workdir "{WORKDIR}" --run {BASE_RUN},ft-{RUN_NAME}-s647,{EVAL_NAME} > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{WORKDIR}/results/summary.md").read()))
import shutil; shutil.copy(f"{WORKDIR}/results/summary.md", f"{WORKDIR}/results/summary_{EVAL_NAME}.md")
print("saved a copy: results/summary_" + EVAL_NAME + ".md")

## 10. Show results again (no GPU needed)
Every evaluation cell writes `results/summary.md`, so the last cell to finish overwrites it (each cell also saves its own copy, `results/summary_<name>.md`). The answers and grades of every run stay on Drive, so this rebuilds the report for any runs you list.

In [ ]:
SHOW = "qwen35-2b-nothink,ft-qwen-v1.2-lora-s647,ft-qwen-v1.2-lora-s647-x50"
!python "{WORKDIR}/physeval.py" report --workdir "{WORKDIR}" --run {SHOW} > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{WORKDIR}/results/summary.md").read()))